# Qwen capability-demand service

This notebook runs the Qwen model behind an authenticated FastAPI /match endpoint. The service accepts only candidate capabilities and job demands.


## 1. Install Kaggle dependencies

Install the runtime packages required by the model server.


In [ ]:
!pip install -U llama-cpp-python
!pip install -q fastapi uvicorn requests

## 2. Load the Qwen model

Load the pinned GGUF model used by the service.


In [ ]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="ccore/Qwen3.5-4B-3D-Coder",
    filename="Qwen3.5-4B-3D-Coder-Q4_K_M.gguf",
    n_ctx=8192,
    verbose=False
)


## 3. Define the authenticated API

Define strict request/response models, the prompt contract, and the /health, /chat, and /match routes.


In [ ]:
import json
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from typing import Any, Literal
from threading import Lock
from kaggle_secrets import UserSecretsClient
import uvicorn
import threading

SECRET_LABEL = "SYNTHETICDATA_API_TOKEN"
try:
    API_TOKEN = UserSecretsClient().get_secret(SECRET_LABEL)
except Exception as exc:
    raise RuntimeError(
        f"Missing Kaggle secret: {SECRET_LABEL}. "
        "Create it through Add-ons > Secrets and attach it to this notebook."
    ) from exc

if not API_TOKEN or not API_TOKEN.strip():
    raise RuntimeError(f"Kaggle secret {SECRET_LABEL} exists but is empty.")
API_TOKEN = API_TOKEN.strip()

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
llm_lock = Lock()

class CapabilityItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    item_code: str
    value: float | None = None
    unit: str | None = None
    method: Literal["self_reported", "tested"] | None = None
    score: int | None = Field(default=None, ge=1, le=5)

class DemandItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    dimension_code: str
    value_state: Literal["known", "unknown"]
    value: Any = None
    unit: str | None = None
    frequency_band: Literal["never", "occasional", "frequent", "constant"] | None = None
    duration_min: float | None = Field(default=None, ge=0)
    essential: bool = False

class MatchRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    candidate_capabilities: list[CapabilityItem]
    job_demands: list[DemandItem]

class MatchResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    dimension_code: str
    status: Literal["Met", "Met with accommodation", "Gap", "Unknown"]
    candidate_item_codes: list[str]
    reason_code: str
    evidence: list[dict[str, Any]]
    confidence: Literal["high", "medium", "low"]

class MatchResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    status: Literal["ok"]
    results: list[MatchResult]

PROMPT_VERSION = "match-system-v1"
SYSTEM_PROMPT = """You are a structured capability-demand analysis component.

Use only candidate_capabilities and job_demands from the request.
Do not use or infer skills, disability, diagnosis, CV text, preferences, age,
gender, persona, or any other fields. Do not invent missing evidence.

Compare only capability items to job demands. Preserve supplied item codes,
values, units, and value states. An unknown demand or missing candidate
counterpart is Unknown. A numeric comparison is Met only when supplied values
and units support it. A failed essential requirement is Gap unless supplied
evidence explicitly supports an accommodation that resolves it. Never infer a
capability from a diagnosis, job title, or general knowledge.

Return one JSON object and no prose or Markdown fences. Return exactly one
result for every job demand. Use only these statuses: Met, Met with
accommodation, Gap, Unknown. Use only these confidence values: high, medium,
low. Every result must contain dimension_code, status, candidate_item_codes,
reason_code, evidence, and confidence. Evidence must identify supplied
candidate capability or job demand items only.
"""
FEW_SHOT_EXAMPLES = [
    (
        {
            "candidate_capabilities": [
                {"item_code": "PC06", "value": 170, "unit": "kg", "method": "tested"}
            ],
            "job_demands": [
                {"dimension_code": "D02", "value_state": "known", "value": 120, "unit": "kg", "essential": True}
            ],
        },
        {
            "status": "ok",
            "results": [{
                "dimension_code": "D02",
                "status": "Met",
                "candidate_item_codes": ["PC06"],
                "reason_code": "candidate_value_meets_demand",
                "evidence": [
                    {"source": "candidate_capabilities", "item_code": "PC06"},
                    {"source": "job_demands", "dimension_code": "D02"},
                ],
                "confidence": "high",
            }],
        },
    ),
    (
        {
            "candidate_capabilities": [
                {"item_code": "PC06", "value": 80, "unit": "kg", "method": "tested"}
            ],
            "job_demands": [
                {"dimension_code": "D02", "value_state": "known", "value": 120, "unit": "kg", "essential": True}
            ],
        },
        {
            "status": "ok",
            "results": [{
                "dimension_code": "D02",
                "status": "Gap",
                "candidate_item_codes": ["PC06"],
                "reason_code": "candidate_value_below_demand",
                "evidence": [
                    {"source": "candidate_capabilities", "item_code": "PC06"},
                    {"source": "job_demands", "dimension_code": "D02"},
                ],
                "confidence": "high",
            }],
        },
    ),
    (
        {
            "candidate_capabilities": [],
            "job_demands": [
                {"dimension_code": "D11", "value_state": "unknown", "value": None, "essential": False}
            ],
        },
        {
            "status": "ok",
            "results": [{
                "dimension_code": "D11",
                "status": "Unknown",
                "candidate_item_codes": [],
                "reason_code": "demand_unknown",
                "evidence": [{"source": "job_demands", "dimension_code": "D11"}],
                "confidence": "low",
            }],
        },
    ),
]


def _match_messages(req: MatchRequest) -> list[dict[str, str]]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for example_input, example_output in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": json.dumps(example_input, ensure_ascii=False),
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(example_output, ensure_ascii=False),
        })
    user = json.dumps({
        "candidate_capabilities": [item.model_dump() for item in req.candidate_capabilities],
        "job_demands": [item.model_dump() for item in req.job_demands],
    }, ensure_ascii=False)
    messages.append({"role": "user", "content": user})
    return messages


def _parse_match_response(answer: str) -> MatchResponse:
    text = answer.strip()
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1].strip()
    if text.startswith("```"):
        text = text.removeprefix("```").removeprefix("json").removesuffix("```").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("Qwen response does not contain a JSON object.")
    return MatchResponse.model_validate(json.loads(text[start:end + 1]))

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str

class ChatRequest(BaseModel):
    messages: list[ChatMessage]
    max_tokens: int = Field(default=800, ge=1, le=2048)
    temperature: float = Field(default=0.2, ge=0, le=1)

@app.get("/health")
def health():
    return {"status": "online"}

@app.post("/chat")
def chat(req: ChatRequest, authorization: str | None = Header(default=None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    messages = [m.model_dump() for m in req.messages]
    with llm_lock:
        result = llm.create_chat_completion(messages=messages, max_tokens=req.max_tokens, temperature=req.temperature)
    return {"answer": result["choices"][0]["message"]["content"]}

@app.post("/match")
def match(req: MatchRequest, authorization: str | None = Header(default=None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    with llm_lock:
        result = llm.create_chat_completion(
            messages=_match_messages(req),
            max_tokens=1200,
            temperature=0.0,
        )
    try:
        return _parse_match_response(result["choices"][0]["message"]["content"]).model_dump()
    except (KeyError, TypeError, ValueError, ValidationError) as exc:
        raise HTTPException(status_code=502, detail="Qwen returned an invalid /match response.") from exc


## 4. Start FastAPI

Start the local FastAPI server on port 8000.


In [ ]:
server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )
)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

## 5. Download Cloudflare Tunnel

Download the temporary Cloudflare Tunnel binary used by the Kaggle runtime.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared

!chmod +x /kaggle/working/cloudflared

## 6. Start the temporary tunnel

Publish the local service and capture the current URL. The URL changes when a new quick tunnel is created.


In [ ]:
import subprocess
import time
import re
from pathlib import Path

log_path = "/kaggle/working/tunnel.log"

log_file = open(log_path, "w")

tunnel_process = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

API_URL = None

for _ in range(60):
    log_text = Path(log_path).read_text(
        encoding="utf-8",
        errors="ignore"
    )

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if match:
        API_URL = match.group(0)
        break

    time.sleep(1)

if API_URL is None:
    raise RuntimeError(
        "Tunnel failed. Check tunnel.log and network access."
    )

print("Your API URL:", API_URL)

## 7. Run local and public smoke tests

Verify health and one capability-demand match locally and through the public tunnel.


In [ ]:
import requests
import time

print("FastAPI Thread:", "RUNNING" if server_thread.is_alive() else "STOPPED")
print("Cloudflare Tunnel:", "RUNNING" if tunnel_process.poll() is None else "STOPPED")

def show_response(label: str, response):
    print(f"{label} status:", response.status_code)
    print(f"{label} content type:", response.headers.get("content-type"))
    try:
        print(f"{label} response:", response.json())
    except ValueError:
        print(f"{label} raw response:", repr(response.text[:4000]))

def get_public_health(url: str):
    last_error = None
    for attempt in range(10):
        try:
            return requests.get(url, timeout=20)
        except requests.RequestException as exc:
            last_error = exc
            if attempt < 9:
                time.sleep(3)
    raise RuntimeError(f"Public API did not become reachable: {last_error}") from last_error

try:
    show_response("Local API", requests.get("http://127.0.0.1:8000/health", timeout=5))
except Exception as e:
    print("Local API transport error:", repr(e))

try:
    show_response("Public API", get_public_health(f"{API_URL.rstrip('/')}/health"))
except Exception as e:
    print("Public API transport error:", repr(e))

match_payload = {
    "candidate_capabilities": [{"item_code": "PC06", "value": 170, "unit": "kg", "method": "tested"}],
    "job_demands": [{"dimension_code": "D02", "value_state": "known", "value": 120, "unit": "kg", "essential": True}],
}
try:
    local_match = requests.post(
        "http://127.0.0.1:8000/match",
        headers={"Authorization": f"Bearer {API_TOKEN}"},
        json=match_payload,
        timeout=180,
    )
    show_response("Local match", local_match)
except Exception as e:
    print("Local match transport error:", repr(e))

try:
    public_match = requests.post(
        f"{API_URL.rstrip('/')}/match",
        headers={"Authorization": f"Bearer {API_TOKEN}"},
        json=match_payload,
        timeout=180,
    )
    show_response("Public match", public_match)
except Exception as e:
    print("Public match transport error:", repr(e))
